<a href="https://colab.research.google.com/github/Joao-VitorGuerra/Collab_AD05/blob/main/Desafios%20Tecnicos/Python_API/Desafio_01_Passagens_Aereas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Instala as bibliotecas usadas no guia.
!pip install -q gspread faker

In [2]:
# Importa as ferramentas para autenticação, geração de dados e sorteios.
import json, gspread
from google.colab import userdata
from faker import Faker
import random

# Lê a credencial salva nos Secrets e conecta à planilha.
info = json.loads(userdata.get("GCP_SA_JSON"))
gc = gspread.service_account_from_dict(info)
# Exibe o e-mail da conta autenticada.
print("Conectado como:", info["client_email"])

Conectado como: desafio1-python@vocal-operand-513817-r2.iam.gserviceaccount.com


In [6]:
# Informe o ID da planilha e o nome que será dado à aba.
SHEET_ID = "1b0jMdJUQS-nZDj-hlrW_zCUpM-Py28O-D_rd3PgRu00"
NOME_TABELA = "05-passagens-aereas"   # nome da tabela na ficha do tema, ex.: "vagas"

# Abre a planilha e seleciona sua primeira aba.
sh = gc.open_by_key(SHEET_ID)
ws = sh.sheet1
# Renomeia a aba e exibe os dados da planilha.
ws.update_title(NOME_TABELA)     # a aba passa a ter o nome da tabela
print(sh.title, "|", ws.title, "|", sh.url)

Desafio_01_Passagens_Aereas | 05-passagens-aereas | https://docs.google.com/spreadsheets/d/1b0jMdJUQS-nZDj-hlrW_zCUpM-Py28O-D_rd3PgRu00


In [7]:
# Substitua pelos nomes EXATOS de coluna da ficha do seu tema
# Define as colunas que formarão o cabeçalho da tabela.
COLUNAS = ["voo_id", "origem", "destino", "companhia_aerea","data_hora_partida","data_hora_chegada","capacidade_assentos"]

# Escreve o cabeçalho na primeira célula e confere o resultado.
ws.update(range_name="A1", values=[COLUNAS])
print("Cabeçalho confere:", ws.row_values(1) == COLUNAS)

Cabeçalho confere: True


In [19]:
# Configura os geradores aleatórios para produzir resultados reproduzíveis.
fake = Faker("pt_BR")
Faker.seed(42)
random.seed(42)

cidades = [
    "São Paulo",
    "Rio de Janeiro",
    "Belo Horizonte",
    "Brasília",
    "Curitiba",
    "Salvador",
    "Recife",
    "Fortaleza",
    "Porto Alegre",
    "Florianópolis",
    "Goiânia",
    "Campinas",
    "Santos",
    "Manaus",
    "Belém"
]




def gerar_linha(i):
    # Gera uma data e hora de partida fictícia

    # Adiciona entre 1 e 8 horas para calcular o horário de chegada


    # Cria um registro fictício para as colunas definidas no esquema.
    return {
        "voo_id": i,
        "origem": random.choice(cidades),
        "destino": random.choice(cidades),
        "companhia_aerea": random.choice(["LATAM", "Gol", "Azul", "Voepass"]),
        "data_hora_partida": fake.date_this_year().isoformat(),
        "data_hora_chegada": fake.date_this_year().isoformat(),
        "capacidade_assentos": random.randint(100, 300)
    }

# Gera 10 registros numerados de 1 a 10.
linhas = [gerar_linha(i) for i in range(1, 11)]

# Organiza os valores na ordem do cabeçalho; nome de coluna incorreto gera KeyError.
valores = [[linha[c] for c in COLUNAS] for linha in linhas]
# Acrescenta os registros à planilha e informa quantos foram enviados.
ws.append_rows(valores, value_input_option="RAW")
print(len(valores), "linhas enviadas")

10 linhas enviadas


In [14]:
# Lê todos os valores da aba para conferir o conteúdo enviado.
valores_lidos = ws.get_all_values()
# Exibe a quantidade de linhas, confere o cabeçalho e mostra exemplos.
print("Linhas (cabeçalho + dados):", len(valores_lidos))    # esperado: 11
print("Cabeçalho confere:", valores_lidos[0] == COLUNAS)
print(valores_lidos[:3])

Linhas (cabeçalho + dados): 11
Cabeçalho confere: True
[['voo_id', 'origem', 'destino', 'companhia_aerea', 'data_hora_partida', 'data_hora_chegada', 'capacidade_assentos'], ['1', 'Alves', 'da Cruz de Goiás', 'LATAM', '2026-02-12', '2026-02-01', '106'], ['2', 'Pastor', 'Andrade do Sul', 'Azul', '2026-01-29', '2026-03-13', '162']]
